## Q2 – Reconstruct payments of 31 Jan at C1 and C2
Setup: pick the catalog/schema written by notebook 03 and the payment day D. The inputs are validated so a typo stops the run. Time zone is UTC so "the day" means the same thing everywhere.

In [0]:
import re
dbutils.widgets.text("catalog", "bdcc")
dbutils.widgets.text("schema", "project_g")
dbutils.widgets.text("payment_day", "2026-01-31")
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
payment_day = dbutils.widgets.get("payment_day").strip()
for value in (catalog, schema):
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value)
assert re.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}", payment_day)
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {schema}")
spark.sql("SET TIME ZONE 'UTC'")

In [0]:
%sql
SHOW TABLES

### Check: each processor/provider reference maps to one payment
Reference mappings are the only link between processor/provider records and gateway payments. If a reference pointed to two payments, its money would be counted twice. Result: 0 rows at both cutoffs, so the join is safe.

In [0]:
%sql
SELECT cutoff_id, from_system, from_reference,
       count(DISTINCT concat_ws('|', gateway_system, gateway_reference)) AS payments,
       collect_set(concat_ws('|', gateway_system, gateway_reference)) AS which_payments
FROM clean_reference_mappings
GROUP BY cutoff_id, from_system, from_reference
HAVING count(DISTINCT concat_ws('|', gateway_system, gateway_reference)) > 1

### Money rows we cannot trust
Collects captures/refunds/postings that are QUARANTINED (conflicting versions) or REJECTED with no valid replacement. Only processor_events has them (86 quarantined, 31 invalid amount, 46 invalid timestamp), and the counts are the same at C1 and C2. Payments touched by these get dq_flag = true.

In [0]:
MONEY = {"processor_events": ("processor_system", "processor_reference"),
         "refund_events":    ("processor_system", "processor_reference"),
         "settlement_lines": ("provider_system",  "provider_reference")}
parts = []
for name, (sys_col, ref_col) in MONEY.items():
    parts.append(f"""
SELECT s.cutoff_id, '{name}' AS source_table, s.{sys_col} AS from_system, s.{ref_col} AS from_reference,
       s.business_key, s.version_no, s.row_status, s.validation_issue, s.source_row_id, s.delivery_id
FROM snap_{name} s
WHERE s.row_status = 'QUARANTINED'
UNION ALL
SELECT s.cutoff_id, '{name}', s.{sys_col}, s.{ref_col},
       s.business_key, s.version_no, s.row_status, s.validation_issue, s.source_row_id, s.delivery_id
FROM snap_{name} s
LEFT ANTI JOIN snap_{name} c
  ON c.cutoff_id = s.cutoff_id AND c.business_key = s.business_key
 AND c.row_status = 'CURRENT' AND c.version_no >= s.version_no
WHERE s.row_status = 'REJECTED'""")
spark.sql("CREATE OR REPLACE VIEW dq_money_rows AS\n" + "\nUNION ALL\n".join(parts))

In [0]:
%sql
SELECT cutoff_id, source_table, row_status, validation_issue, count(*) AS n_rows
FROM dq_money_rows GROUP BY ALL ORDER BY ALL

### Payment close per cutoff
One row per cutoff × payment × currency. Expected = captures − reversals − completed refunds; settled = linked bank postings. Status: MATCHED, PENDING (not due yet), OVERDUE, OVERSETTLED, NOTHING_EXPECTED, and UNLINKED for postings with no payment. Every join matches on cutoff so C1 and C2 never mix.

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE q2_payment_close AS
WITH
pay AS (
  SELECT g.cutoff_id, g.cutoff_ts, g.gateway_system, g.gateway_reference, g.business_key AS payment_key,
         g.currency_cd AS requested_currency, coalesce(t.acquiring_bank_id, 'UNKNOWN') AS bank_id
  FROM clean_gateway_requests g
  LEFT JOIN clean_terminal_assignments t
    ON t.cutoff_id = g.cutoff_id AND t.terminal_system = g.terminal_system AND t.terminal_id = g.terminal_id
   AND g.request_timestamp >= t.valid_from AND g.request_timestamp < t.valid_to
  WHERE to_date(g.request_timestamp) = DATE'{payment_day}'
),
map AS (
  SELECT DISTINCT cutoff_id, from_system, from_reference, gateway_system, gateway_reference
  FROM clean_reference_mappings
),
pe AS (
  SELECT p.*, m.gateway_system, m.gateway_reference
  FROM clean_processor_events p
  JOIN map m ON m.cutoff_id = p.cutoff_id AND m.from_system = p.processor_system
            AND m.from_reference = p.processor_reference
),
caps AS (SELECT * FROM pe WHERE event_type = 'CAPTURE'),
movements AS (
  SELECT cutoff_id, gateway_system, gateway_reference, currency_cd, amount_minor AS amt FROM caps
  UNION ALL
  SELECT r.cutoff_id, r.gateway_system, r.gateway_reference, r.currency_cd, -r.amount_minor
  FROM pe r
  JOIN caps c ON c.cutoff_id = r.cutoff_id AND c.processor_system = r.processor_system
             AND c.event_id = r.related_event_id
  WHERE r.event_type = 'REVERSAL'
  UNION ALL
  SELECT f.cutoff_id, m.gateway_system, m.gateway_reference, f.currency_cd, -f.amount_minor
  FROM clean_refund_events f
  JOIN map m ON m.cutoff_id = f.cutoff_id AND m.from_system = f.processor_system
            AND m.from_reference = f.processor_reference
  WHERE f.refund_status = 'COMPLETED'
),
expected AS (
  SELECT cutoff_id, gateway_system, gateway_reference, currency_cd, sum(amt) AS expected_minor
  FROM movements GROUP BY ALL
),
due AS (
  SELECT cutoff_id, gateway_system, gateway_reference, max(settlement_due_at) AS settlement_due_at
  FROM caps GROUP BY ALL
),
st AS (
  SELECT s.*, m.gateway_system, m.gateway_reference
  FROM clean_settlement_lines s
  LEFT JOIN map m ON m.cutoff_id = s.cutoff_id AND m.from_system = s.provider_system
                 AND m.from_reference = s.provider_reference
),
settled AS (
  SELECT cutoff_id, gateway_system, gateway_reference, currency_cd, sum(signed_amount_minor) AS settled_minor
  FROM st WHERE gateway_reference IS NOT NULL GROUP BY ALL
),
amounts AS (
  SELECT cutoff_id, gateway_system, gateway_reference, currency_cd,
         coalesce(e.expected_minor, 0) AS expected_minor, coalesce(s.settled_minor, 0) AS settled_minor
  FROM expected e FULL JOIN settled s USING (cutoff_id, gateway_system, gateway_reference, currency_cd)
),
dq AS (
  SELECT DISTINCT d.cutoff_id, m.gateway_system, m.gateway_reference
  FROM dq_money_rows d
  JOIN map m ON m.cutoff_id = d.cutoff_id AND m.from_system = d.from_system AND m.from_reference = d.from_reference
),
linked AS (
  SELECT p.cutoff_id, p.cutoff_ts, p.payment_key, p.bank_id,
         coalesce(a.currency_cd, p.requested_currency) AS currency_cd,
         coalesce(a.expected_minor, 0) AS expected_minor, coalesce(a.settled_minor, 0) AS settled_minor,
         d.settlement_due_at, q.gateway_reference IS NOT NULL AS dq_flag
  FROM pay p
  LEFT JOIN amounts a ON a.cutoff_id = p.cutoff_id AND a.gateway_system = p.gateway_system
                     AND a.gateway_reference = p.gateway_reference
  LEFT JOIN due d ON d.cutoff_id = p.cutoff_id AND d.gateway_system = p.gateway_system
                 AND d.gateway_reference = p.gateway_reference
  LEFT JOIN dq q ON q.cutoff_id = p.cutoff_id AND q.gateway_system = p.gateway_system
                AND q.gateway_reference = p.gateway_reference
)
SELECT cutoff_id, DATE'{payment_day}' AS payment_day, payment_key, bank_id, currency_cd,
       expected_minor, settled_minor, expected_minor - settled_minor AS outstanding_minor,
       CASE WHEN expected_minor = 0 AND settled_minor = 0 THEN 'NOTHING_EXPECTED'
            WHEN expected_minor - settled_minor < 0 THEN 'OVERSETTLED'
            WHEN expected_minor - settled_minor = 0 THEN 'MATCHED'
            WHEN cutoff_ts < settlement_due_at THEN 'PENDING'
            ELSE 'OVERDUE' END AS status,
       settlement_due_at, dq_flag
FROM linked
UNION ALL
SELECT cutoff_id, DATE'{payment_day}', concat_ws('|', provider_system, provider_reference),
       coalesce(bank_id, 'UNKNOWN'), currency_cd,
       0, sum(signed_amount_minor), -sum(signed_amount_minor), 'UNLINKED', CAST(NULL AS TIMESTAMP), false
FROM st
WHERE gateway_reference IS NULL AND booking_date = DATE'{payment_day}'
GROUP BY cutoff_id, provider_system, provider_reference, bank_id, currency_cd
""")

In [0]:
%sql
SELECT cutoff_id, status, count(*) AS payments, count_if(dq_flag) AS dq
FROM q2_payment_close GROUP BY ALL ORDER BY ALL

### 5. Grain check of the result
Every Q2 total is a sum over `q2_payment_close`, so a duplicated payment would inflate the totals with no error.
We check: one row per cutoff, payment and currency; whether any payment has money in two currencies; and that every payment of D appears at both cutoffs.

In [0]:
dupes = spark.sql("""
SELECT count(*) AS n FROM (
  SELECT cutoff_id, payment_key, currency_cd FROM q2_payment_close
  GROUP BY ALL HAVING count(*) > 1)""").first()["n"]
assert dupes == 0, "a payment appears twice at one cutoff"

multi_ccy = spark.sql("""
SELECT count(*) AS n FROM (
  SELECT cutoff_id, payment_key FROM q2_payment_close WHERE status <> 'UNLINKED'
  GROUP BY ALL HAVING count(DISTINCT currency_cd) > 1)""").first()["n"]
print("payments with movements in more than one currency:", multi_ccy)

display(spark.sql("""
SELECT cutoff_id, count(*) AS payments FROM q2_payment_close
WHERE status <> 'UNLINKED' GROUP BY cutoff_id ORDER BY cutoff_id"""))

### 6. Q2 answer: expected, settled and outstanding by bank and currency
For payment day D = 2026-01-31, at C1 and C2. Currency is part of the grouping because EUR and GBP must never be added together.
Amounts are kept in minor units; the `/100` columns are only for readability (both currencies have 100 minor units).
The status counts show why money is outstanding: not due yet (PENDING), late (OVERDUE), too much (OVERSETTLED), or not linked to any payment (UNLINKED).

In [0]:
%sql
SELECT cutoff_id, bank_id, currency_cd,
       sum(expected_minor)    AS expected_minor,
       sum(settled_minor)     AS settled_minor,
       sum(outstanding_minor) AS outstanding_minor,
       round(sum(expected_minor) / 100.0, 2)    AS expected,
       round(sum(settled_minor) / 100.0, 2)     AS settled,
       round(sum(outstanding_minor) / 100.0, 2) AS outstanding,
       count_if(status = 'MATCHED')     AS matched,
       count_if(status = 'PENDING')     AS pending,
       count_if(status = 'OVERDUE')     AS overdue,
       count_if(status = 'OVERSETTLED') AS oversettled,
       count_if(status = 'UNLINKED')    AS unlinked,
       count_if(dq_flag)                AS dq_cases
FROM q2_payment_close
GROUP BY ALL
ORDER BY bank_id, currency_cd, cutoff_id

### What changed between C1 and C2
- **Settled went up** (EUR +3,995.22, GBP +417.67): provider postings delivered after C1. 73 payments moved from PENDING to MATCHED.
- **Expected went down a little** (EUR −715.90, GBP −32.31): reversals and completed refunds delivered after C1.
- **The other 95 PENDING payments became OVERDUE** (86 EUR, 9 GBP): their settlement due time passed before C2 and no posting arrived. These are the real open items for Finance.
- **UNLINKED went from 19 to 14:** new reference mappings after C1 linked 5 postings to their payment.
- **OVERSETTLED (37) and dq cases (15) are the same at both cutoffs.** No later delivery fixed them. 15 are caused by rejected or quarantined processor rows, so their number cannot be signed off.
- **GBP is net oversettled at C2 (−193.11),** mostly BANK03 (4 oversettled + 5 unlinked postings). Bank totals include UNLINKED postings, counted as settled with nothing expected.
- BANK05 EUR is fully matched at both cutoffs.

In [0]:
%sql
SHOW TABLES LIKE 'raw*|stg*|snap*'

### 7. Choose a payment to trace
A payment whose status or settled amount changes between C1 and C2 shows why the snapshots differ.

In [0]:
%sql
SELECT c1.payment_key, c1.bank_id, c1.currency_cd,
       c1.status AS status_c1, c2.status AS status_c2,
       c1.settled_minor AS settled_c1, c2.settled_minor AS settled_c2,
       c1.dq_flag AS dq_c1, c2.dq_flag AS dq_c2
FROM q2_payment_close c1
JOIN q2_payment_close c2
  ON c2.payment_key = c1.payment_key AND c2.currency_cd = c1.currency_cd AND c2.cutoff_id = 'C2'
WHERE c1.cutoff_id = 'C1' AND c1.status <> 'UNLINKED'
  AND (c1.status <> c2.status OR c1.settled_minor <> c2.settled_minor)
ORDER BY c1.payment_key
LIMIT 20

### 8. Trace one payment from raw records through both snapshots
We list every processor/provider reference that any mapping (any cutoff, any version) links to this payment.
We use `stg_` instead of `clean_` because the trace must also show rejected, late and superseded rows.

In [0]:
gsys, gref = "GW01", "000039372"
payment_key = f"{gsys}|{gref}"
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW trace_refs AS
SELECT DISTINCT from_system, from_reference
FROM stg_reference_mappings
WHERE gateway_system = '{gsys}' AND gateway_reference = '{gref}'
""")
display(spark.table("trace_refs"))

### 9. The gateway request and its mapping: raw and in both snapshots

In [0]:
display(spark.sql(f"""
SELECT 'raw' AS layer, NULL AS cutoff_id, NULL AS row_status, source_row_id, delivery_id,
       request_timestamp, currency_cd, requested_amount_minor, terminal_system, terminal_id
FROM raw_gateway_requests WHERE gateway_system = '{gsys}' AND gateway_reference = '{gref}'
UNION ALL
SELECT 'snapshot', cutoff_id, row_status, source_row_id, delivery_id,
       request_timestamp, currency_cd, requested_amount_minor, terminal_system, terminal_id
FROM snap_gateway_requests WHERE business_key = '{payment_key}'
ORDER BY layer, cutoff_id, source_row_id"""))

display(spark.sql(f"""
SELECT cutoff_id, business_key, version_no, row_status, from_system, from_reference,
       effective_at, delivered_at, source_row_id
FROM snap_reference_mappings WHERE gateway_system = '{gsys}' AND gateway_reference = '{gref}'
ORDER BY cutoff_id, business_key, version_no"""))

### 10. Every money row for this payment: raw and in both snapshots
A row that is in raw but missing at a cutoff was delivered after that cutoff.
Only `CURRENT` rows count; `NOT_YET_EFFECTIVE`, `SUPERSEDED`, `REPLAY`, `REJECTED` and `QUARANTINED` rows do not.

In [0]:
TRACE = {
    "processor_events": ("processor_system", "processor_reference", "event_id",   "event_type",    "amount_minor"),
    "refund_events":    ("processor_system", "processor_reference", "refund_id",  "refund_status", "amount_minor"),
    "settlement_lines": ("provider_system",  "provider_reference",  "posting_id", "posting_type",  "signed_amount_minor"),
}
for name, (sys_col, ref_col, id_col, kind, amount) in TRACE.items():
    print(name)
    display(spark.sql(f"""
SELECT 'raw' AS layer, NULL AS cutoff_id, NULL AS row_status, r.source_row_id, r.delivery_id,
       concat_ws('|', r.{sys_col}, r.{id_col}) AS business_key,
       r.version, r.{kind} AS kind, r.{amount} AS amount, r.effective_at
FROM raw_{name} r JOIN trace_refs t ON t.from_system = r.{sys_col} AND t.from_reference = r.{ref_col}
UNION ALL
SELECT 'snapshot', s.cutoff_id, s.row_status, s.source_row_id, s.delivery_id,
       s.business_key, s.version, s.{kind}, s.{amount}, s.effective_at
FROM snap_{name} s JOIN trace_refs t ON t.from_system = s.{sys_col} AND t.from_reference = s.{ref_col}
ORDER BY layer, cutoff_id, business_key, version, source_row_id"""))

### 11. Result for this payment at C1 and C2
Every number here must be explainable from the rows above.

In [0]:
display(spark.sql(f"SELECT * FROM q2_payment_close WHERE payment_key = '{payment_key}' ORDER BY cutoff_id"))

### 12. Trace result: GW01|000039372 (BANK03, EUR)
**Raw evidence**
- Authorisation `PR01|E000039372A` v1, 17.89 (row 1156, delivery 2026-02-01 06:00).
- Capture `PR01|E000039372C1` v1, **19.09** (row 1157, delivery 2026-02-01 06:00).
- Capture `PR01|E000039372C1` v2, **17.89**, same effective_at (row 1158, delivery **2026-02-02 07:00**), a correction of the captured amount.
- Settlement posting `ST01|P0000393721` v1, 17.89 (row 625, delivery 2026-02-01 06:00).
- No refunds.

**At C1:** capture v2 had not been delivered yet, so v1 (19.09) is CURRENT. Expected 19.09, settled 17.89, outstanding 1.20. The due time (2026-02-02 08:00) is after C1, so the status is **PENDING**.

**At C2:** v2 was delivered, so it becomes CURRENT and v1 becomes SUPERSEDED. Expected 17.89 = settled 17.89, so the status is **MATCHED**.

**Why the snapshots differ:** a late *correction* of the capture, not a new payment movement. The authorisation is not counted (it moves no money). Without the correction, this payment would have been **OVERDUE** at C2, since its due time equals the C2 cutoff. This shows why each cutoff must keep only the versions delivered by then: rebuilding C1 with today's data would hide that the 1.20 gap ever existed.

### Assumptions and limits (Q2)
1. **Payment day** D = 2026-01-31 is the UTC date of `request_timestamp`. Cutoffs: C1 = 2026-02-01 08:00 UTC, C2 = 2026-02-02 08:00 UTC.
2. **Only CURRENT facts count** at each cutoff: delivered and effective by the cutoff, highest version, no conflict, replays collapsed (notebook 03). We never fall back to an older version.
3. **Bank** of a payment = `acquiring_bank_id` of the terminal assignment valid at request time (`valid_from` inclusive, `valid_to` exclusive); `UNKNOWN` if none matches. Bank of an UNLINKED posting = its own `bank_id`, or `UNKNOWN`.
4. **Expected** = captures − reversals of a known capture − COMPLETED refunds. REQUESTED refunds do not reduce it yet. **Settled** = sum of signed provider postings (receipts +, returns −).
5. **Status:** nothing expected and nothing settled → NOTHING_EXPECTED; outstanding < 0 → OVERSETTLED; = 0 → MATCHED; > 0 and cutoff < `settlement_due_at` → PENDING; otherwise OVERDUE (due exactly at the cutoff counts as overdue).
6. **Data quality:** a payment touched by a QUARANTINED row, or by a REJECTED row with no valid same-or-higher version, gets `dq_flag = true`. Its number is shown but not signed off (15 payments at both cutoffs).
7. **Postings with no mapping** are kept as UNLINKED and included in the bank totals, never dropped.
8. **Amounts** stay in minor units (EUR and GBP both have 100). Currencies are never added together.
9. Every reference maps to exactly one payment per cutoff (Cell 2 returned no rows), so the mapping join cannot double-count.